# 02 chunking and vector index

**Owner:** Daniel. Original notebook cells: 68, 69.

Run in the shared Databricks session using `run_pipeline.ipynb` or the documented stage runner. This is not a standalone clean-kernel notebook. Saved outputs have been removed for public-repository safety. See `docs/execution_order.md`.

## Overlapping Chunking, Vector Index & Grounded ask_rag() Prototype

In [ ]:
# RAG Chunking, Vector Index, Grounded Retriever & Website-Ready ask_rag()

import os
import json
import numpy as np
import pandas as pd
from databricks.sdk import WorkspaceClient

TEAM_VOLUME = "/Volumes/workspace/default/eye_style_team2"
RAG_DATA_DIR = f"{TEAM_VOLUME}/rag"
MANIFEST_PATH = f"{RAG_DATA_DIR}/rag_corpus_manifest.json"

# ---------------------------------------------------------------------
# 1. Load Approved Corpus from Unity Catalog Volume
# ---------------------------------------------------------------------
with open(MANIFEST_PATH, "r", encoding="utf-8") as f:
    rag_manifest = json.load(f)

loaded_docs = []
for entry in rag_manifest["documents"]:
    with open(entry["file_path"], "r", encoding="utf-8") as f:
        loaded_docs.append(json.load(f))

# ---------------------------------------------------------------------
# 2. Overlapping Word-Window Chunker Preserving Required Metadata
# ---------------------------------------------------------------------
def build_rag_chunks(documents, chunk_size_words=95, chunk_overlap_words=25):
    if chunk_overlap_words >= chunk_size_words:
        raise ValueError("chunk_overlap_words must be smaller than chunk_size_words.")

    chunks = []
    step = chunk_size_words - chunk_overlap_words

    for doc in documents:
        for sec_idx, sec in enumerate(doc["sections"], start=1):
            words = sec["text"].split()
            if len(words) <= chunk_size_words:
                windows = [words]
            else:
                windows = []
                for start in range(0, len(words), step):
                    window = words[start : start + chunk_size_words]
                    if window:
                        windows.append(window)
                    if start + chunk_size_words >= len(words):
                        break

            for part_idx, window in enumerate(windows, start=1):
                chunk_id = f"{doc['doc_id']}-S{sec_idx}-C{part_idx}"
                chunk_text = " ".join(window)
                chunks.append({
                    "chunk_id": chunk_id,
                    "doc_id": doc["doc_id"],
                    "source_title": doc["source_title"],
                    "source_type": doc["source_type"],
                    "version_date": doc["version_date"],
                    "section": sec["section"],
                    "source_ref": f"{doc['doc_id']} | {doc['source_title']} — {sec['section']}",
                    "text": chunk_text,
                    "word_count": len(window),
                })
    return chunks

# ---------------------------------------------------------------------
# 3. Embedding Index (Dense SentenceTransformers with TF-IDF Fallback)
# ---------------------------------------------------------------------
class OptometryRAGIndex:
    def __init__(self, chunks, embedding_model_name="sentence-transformers/all-MiniLM-L6-v2"):
        self.chunks = chunks
        self.embedding_model_name = embedding_model_name
        self.backend = None
        self.model = None
        self.vectorizer = None
        self.matrix = None

        # Prepend title & section for stronger semantic alignment
        texts = [
            f"{c['source_title']} | {c['section']}: {c['text']}"
            for c in chunks
        ]

        try:
            from sentence_transformers import SentenceTransformer
            self.model = SentenceTransformer(embedding_model_name)
            self.matrix = np.asarray(
                self.model.encode(texts, normalize_embeddings=True, show_progress_bar=False),
                dtype=np.float32,
            )
            self.backend = f"dense:{embedding_model_name}"
        except Exception as exc:
            # Graceful fallback if workspace blocks external model downloads
            from sklearn.feature_extraction.text import TfidfVectorizer
            self.vectorizer = TfidfVectorizer(ngram_range=(1, 2), stop_words="english")
            self.matrix = self.vectorizer.fit_transform(texts)
            self.backend = f"tfidf_fallback ({type(exc).__name__})"

    def retrieve(self, query, top_k=3):
        if self.backend.startswith("dense:"):
            q_vec = np.asarray(
                self.model.encode([query], normalize_embeddings=True, show_progress_bar=False),
                dtype=np.float32,
            )[0]
            scores = self.matrix @ q_vec
        else:
            from sklearn.metrics.pairwise import cosine_similarity
            q_vec = self.vectorizer.transform([query])
            scores = cosine_similarity(q_vec, self.matrix).ravel()

        top_indices = np.argsort(scores)[::-1][:top_k]
        results = []
        for idx in top_indices:
            item = dict(self.chunks[int(idx)])
            item["similarity"] = round(float(scores[int(idx)]), 4)
            results.append(item)
        return results

# Build default baseline index (before Step 3 parameter tuning)
RAG_CONFIG = {
    "chunk_size_words": 95,
    "chunk_overlap_words": 25,
    "top_k": 3,
    "min_similarity_dense": 0.35,
    "min_similarity_tfidf": 0.12,
}

rag_chunks = build_rag_chunks(
    loaded_docs,
    chunk_size_words=RAG_CONFIG["chunk_size_words"],
    chunk_overlap_words=RAG_CONFIG["chunk_overlap_words"],
)
rag_index = OptometryRAGIndex(rag_chunks)

# Save chunk table to Unity Catalog Volume for reproducibility
with open(f"{RAG_DATA_DIR}/rag_chunks_v1.json", "w", encoding="utf-8") as f:
    json.dump(rag_chunks, f, indent=2)

print(f"RAG index ready | Backend: {rag_index.backend} | Chunks: {len(rag_chunks)}")

# ---------------------------------------------------------------------
# 4. Databricks LLM Client & Grounded RAG Generator (Thomas-Compatible)
# ---------------------------------------------------------------------
def _get_rag_llm_client():
    if "client" in globals() and globals()["client"] is not None:
        return globals()["client"], globals().get("MODEL") or "system.ai.gpt-oss-120b"
    ws = WorkspaceClient()
    llm_client = ws.serving_endpoints.get_open_ai_client()
    llm_client.base_url = ws.config.host.rstrip("/") + "/ai-gateway/mlflow/v1"
    return llm_client, "system.ai.gpt-oss-120b"

def _extract_llm_text(content):
    if content is None:
        return ""
    if isinstance(content, str):
        return content.strip()
    if isinstance(content, list):
        parts = [
            b.get("text", "")
            for b in content
            if isinstance(b, dict) and b.get("type") in ("text", "output_text")
        ]
        return "\n".join(p for p in parts if p).strip()
    return str(content).strip()

ABSTENTION_PREFIX = "Insufficient evidence in the approved Eye Style knowledge corpus"

RAG_SYSTEM_PROMPT = f"""
You are the Eye Style Optometry RAG Knowledge Assistant.
Answer staff operational questions STRICTLY using the retrieved context passages provided below.

RULES:
1. Only state facts directly supported by the provided context passages. Do not use outside knowledge, do not extrapolate, and do not invent clinic policies, prices, medical advice, or live patient data.
2. Cite the exact doc_id and section in brackets for every claim, e.g. [DOC-05: Lab Turnaround Times & Overdue Escalation].
3. If the retrieved context does not contain enough information to answer the question directly, you MUST abstain by responding with:
   "{ABSTENTION_PREFIX} to answer this question." followed by one brief sentence stating what information is missing.
4. Never provide clinical diagnoses, prescribe treatment, claim live VSP/Crystal PM access, or claim to have modified records or contacted patients.
5. Keep answers concise, accurate, and practical for optometry staff.
""".strip()

RAG_ALLOWED_ROLES = {
    "owner": "Owner",
    "receptionist": "Front Desk",
    "front desk": "Front Desk",
    "optician": "Optician",
}

def ask_rag(
    question,
    authenticated_role="Owner",
    authenticated_subject="demo-rag-user",
    index=None,
    top_k=None,
    min_similarity=None,
):
    if not isinstance(question, str) or not question.strip():
        raise ValueError("Question cannot be empty.")

    role_key = str(authenticated_role).strip().lower()
    if role_key not in RAG_ALLOWED_ROLES:
        raise ValueError("Unauthorized user role.")
    role = RAG_ALLOWED_ROLES[role_key]

    active_index = index or rag_index
    k = top_k or RAG_CONFIG["top_k"]
    threshold = (
        min_similarity
        if min_similarity is not None
        else (
            RAG_CONFIG["min_similarity_dense"]
            if active_index.backend.startswith("dense:")
            else RAG_CONFIG["min_similarity_tfidf"]
        )
    )

    retrieved = active_index.retrieve(question.strip(), top_k=k)
    top_score = retrieved[0]["similarity"] if retrieved else 0.0

    # Gate 1: Similarity threshold abstention
    if not retrieved or top_score < threshold:
        answer = (
            f"{ABSTENTION_PREFIX} to answer this question. "
            "No approved SOP or reference passage met the minimum retrieval relevance threshold."
        )
        return {
            "success": True,
            "answer": answer,
            "workflow": "RAG",
            "role": role,
            "reports": [],
            "sources": [],
            "retrieved_chunks": retrieved,
            "abstained": True,
            "top_similarity": top_score,
            "notices": ["Abstained due to insufficient supporting evidence in approved corpus."],
            "mode": "demo",
            "requires_staff_review": True,
        }

    # Format context passages for grounded generation
    context_blocks = []
    for i, ch in enumerate(retrieved, start=1):
        context_blocks.append(
            f"[Passage {i} | {ch['doc_id']} | {ch['source_title']} | Section: {ch['section']} | "
            f"Version: {ch['version_date']} | Score: {ch['similarity']}]\n{ch['text']}"
        )
    context_str = "\n\n".join(context_blocks)

    llm_client, model_name = _get_rag_llm_client()
    response = llm_client.chat.completions.create(
        model=model_name,
        messages=[
            {"role": "system", "content": RAG_SYSTEM_PROMPT},
            {
                "role": "user",
                "content": (
                    f"Staff Role: {role}\n\n"
                    f"Approved Context Passages:\n{context_str}\n\n"
                    f"Staff Question: {question.strip()}"
                ),
            },
        ],
        temperature=0.0,
        max_tokens=1800,
    )

    answer = _extract_llm_text(response.choices[0].message.content)
    abstained = ABSTENTION_PREFIX.lower() in answer.lower()

    # Record cited sources (empty if abstained)
    cited_sources = (
        []
        if abstained
        else list(dict.fromkeys(ch["source_ref"] for ch in retrieved if ch["doc_id"] in answer))
    )
    # Fallback to top retrieved source if LLM grounded without repeating doc_id token
    if not abstained and not cited_sources and retrieved:
        cited_sources = [retrieved[0]["source_ref"]]

    return {
        "success": True,
        "answer": answer,
        "workflow": "RAG",
        "role": role,
        "reports": [],
        "sources": cited_sources,
        "retrieved_chunks": retrieved,
        "abstained": abstained,
        "top_similarity": top_score,
        "notices": (
            ["Abstained due to insufficient supporting evidence in approved corpus."]
            if abstained
            else ["Grounded strictly in approved non-PHI Eye Style capstone corpus."]
        ),
        "mode": "demo",
        "requires_staff_review": True,
    }

# ---------------------------------------------------------------------
# 5. Smoke Test: 1 In-Corpus Question + 1 Out-of-Corpus Abstention Test
# ---------------------------------------------------------------------
print("\n=== SMOKE TEST 1: IN-CORPUS SOP QUESTION ===")
res_in = ask_rag(
    "What should an optician do first when a frame order is overdue in fulfillment?",
    authenticated_role="optician",
    authenticated_subject="demo-optician",
)
print("Abstained:", res_in["abstained"], "| Top Similarity:", res_in["top_similarity"])
print("Sources:", res_in["sources"])
print("Answer:\n", res_in["answer"])

print("\n=== SMOKE TEST 2: OUT-OF-CORPUS (MUST ABSTAIN) ===")
res_out = ask_rag(
    "What is the exact 2026 cash price for Zeiss SmartLife progressive lenses and what antibiotic drops treat pink eye?",
    authenticated_role="front desk",
    authenticated_subject="demo-front-desk",
)
print("Abstained:", res_out["abstained"], "| Top Similarity:", res_out["top_similarity"])
print("Sources:", res_out["sources"])
print("Answer:\n", res_out["answer"])